# W&B Registry — Promote the Synthetic Dataset

The **W&B Registry** is an organisation-level catalogue of
promotable artifacts. It sits on top of project-scoped artifacts:
you `log_artifact` inside a project as usual, then `link_artifact`
into a Registry **Collection** to make the artifact org-visible,
aliasable (`production`, `staging`, ...), and discoverable across
projects.

The workshop's Registry is called **`airfoil_surrogates`**. This
notebook adds a **`datasets`** Collection to it, containing the
synthetic NACA airfoil cases that both `experiment_tracking.ipynb`
and `sweeps.ipynb` consume.

**What this notebook does**
1. Generate a small synthetic NACA airfoil dataset (~80 cases).
2. Serialise it to a JSON file.
3. Log it as a project artifact, then `link_artifact` it into the
   Registry's `datasets` Collection with `production` + `v1` aliases.
4. Read it back from the Registry as a sanity check.

**Prerequisites**
- `uv sync`, `uv run wandb login`.
- Access to the `wandb_emea` entity — or edit the `ENTITY` constant
  in §1.

**Run order in the workshop**
`registry.ipynb` (this notebook) → `experiment_tracking.ipynb` →
`sweeps.ipynb` → `reports.ipynb`.


## 1. Setup

Third-party imports and section-local constants.


In [1]:
from __future__ import annotations

import json
import math
import tempfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import wandb

ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

REGISTRY_NAME = "airfoil_surrogates"
DATASET_COLLECTION = "datasets"
DATASET_ARTIFACT_NAME = "synthetic_airfoil_slice"

SEED = 42
N_CASES = 80
N_POINTS = 512

print(f"wandb   = {wandb.__version__}")
print(f"target  = {ENTITY}/{PROJECT}")
print(f"Registry path -> wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}")


wandb   = 0.28.0
target  = wandb_emea/jb_physics_demo
Registry path -> wandb-registry-airfoil_surrogates/datasets


## 2. Synthetic NACA dataset generator

Small, laptop-friendly toy dataset — a 2D neural-field CFD problem.
Each case is a NACA-4-digit airfoil under a sampled
`(angle_of_attack, Reynolds_number)`, with analytic pressure and
velocity fields evaluated at random points in the flow domain.
Singularities (`1/r²` doublet, `1/r` circulation) are clipped so the
targets stay in a physical range.

Features per point: `(x, y, sdf, aoa, log_re)`.
Targets per point: `(pressure, u, v)`.


In [2]:
@dataclass
class AirfoilCase:
    points: np.ndarray    # (N, 2)
    sdf: np.ndarray       # (N,)
    aoa: float
    re: float
    pressure: np.ndarray  # (N,)
    velocity: np.ndarray  # (N, 2)

    def features(self) -> np.ndarray:
        n = self.points.shape[0]
        aoa_col = np.full((n, 1), self.aoa, dtype=np.float32)
        re_col = np.full((n, 1), np.log10(self.re) - 6.0, dtype=np.float32)
        return np.concatenate(
            [self.points.astype(np.float32),
             self.sdf[:, None].astype(np.float32),
             aoa_col, re_col],
            axis=1,
        )

    def targets(self) -> np.ndarray:
        return np.concatenate(
            [self.pressure[:, None], self.velocity], axis=1
        ).astype(np.float32)


def _naca4_camber(x: np.ndarray, m: float, p: float) -> np.ndarray:
    yc = np.zeros_like(x)
    if p > 0:
        mask = x < p
        yc[mask] = (m / p ** 2) * (2 * p * x[mask] - x[mask] ** 2)
        yc[~mask] = (m / (1 - p) ** 2) * ((1 - 2 * p) + 2 * p * x[~mask] - x[~mask] ** 2)
    return yc


def _naca4_thickness(x: np.ndarray, t: float) -> np.ndarray:
    return (t / 0.2) * (
        0.2969 * np.sqrt(np.clip(x, 1e-6, None))
        - 0.1260 * x
        - 0.3516 * x ** 2
        + 0.2843 * x ** 3
        - 0.1015 * x ** 4
    )


def _sample_case(rng: np.random.Generator, n_points: int) -> AirfoilCase:
    m = rng.uniform(0.0, 0.06)
    p = rng.uniform(0.2, 0.6)
    t = rng.uniform(0.08, 0.18)
    aoa = rng.uniform(-8.0, 12.0) * np.pi / 180.0
    re = 10 ** rng.uniform(6.0, 7.0)

    pts = rng.uniform([-0.5, -0.8], [1.5, 0.8], size=(n_points, 2)).astype(np.float32)

    x_chord = np.linspace(0.01, 1.0, 200)
    yc = _naca4_camber(x_chord, m, p)
    yt = _naca4_thickness(x_chord, t)
    upper = np.stack([x_chord, yc + yt], axis=1)
    lower = np.stack([x_chord, yc - yt], axis=1)
    surface = np.concatenate([upper, lower[::-1]], axis=0)

    diffs = pts[:, None, :] - surface[None, :, :]
    dists = np.linalg.norm(diffs, axis=-1)
    sdf = dists.min(axis=1).astype(np.float32)

    r = np.linalg.norm(pts - np.array([0.5, 0.0]), axis=1)
    r = np.maximum(r, 0.1)                     # clip 1/r^2 singularity
    theta = np.arctan2(pts[:, 1], pts[:, 0] - 0.5)
    u_inf = np.cos(aoa)
    v_inf = np.sin(aoa)
    circulation = -2 * np.pi * np.sin(aoa) * (1.0 + m * 5.0)
    doublet = 0.25 * t
    u = u_inf - doublet * np.cos(2 * theta) / r ** 2 - circulation * np.sin(theta) / (2 * np.pi * r)
    v = v_inf - doublet * np.sin(2 * theta) / r ** 2 + circulation * np.cos(theta) / (2 * np.pi * r)
    u = np.clip(u, -3.0, 3.0)
    v = np.clip(v, -3.0, 3.0)
    pressure = 1.0 - (u ** 2 + v ** 2)
    pressure = np.clip(pressure, -3.0, 1.0)

    return AirfoilCase(
        points=pts,
        sdf=sdf,
        aoa=float(aoa),
        re=float(re),
        pressure=pressure.astype(np.float32),
        velocity=np.stack([u, v], axis=1).astype(np.float32),
    )


def synthetic_dataset(n_cases: int, n_points: int = 512, seed: int = 0) -> list[AirfoilCase]:
    rng = np.random.default_rng(seed)
    return [_sample_case(rng, n_points) for _ in range(n_cases)]


## 3. Generate the dataset and serialise it

We generate the cases deterministically (via `seed`) and write them
to a JSON file. The schema is self-describing so any downstream
notebook — Python or otherwise — can load it without extra tooling.


In [3]:
cases = synthetic_dataset(N_CASES, n_points=N_POINTS, seed=SEED)

sample = cases[0]
print(f"total cases: {len(cases)}    per-case: N={N_POINTS}, features={sample.features().shape[-1]}, targets={sample.targets().shape[-1]}")
print(f"first case: aoa={math.degrees(sample.aoa):.2f} deg,  Re={sample.re:.1f}")

tmp_dir = Path(tempfile.mkdtemp())
dataset_path = tmp_dir / f"{DATASET_ARTIFACT_NAME}.json"

payload = {
    "meta": {
        "n_cases": N_CASES,
        "n_points": N_POINTS,
        "seed": SEED,
        "generator": "synthetic NACA-4-digit airfoils",
    },
    "cases": [
        {
            "aoa": float(c.aoa),
            "re": float(c.re),
            "points": c.points.tolist(),
            "sdf": c.sdf.tolist(),
            "pressure": c.pressure.tolist(),
            "velocity": c.velocity.tolist(),
        }
        for c in cases
    ],
}

with open(dataset_path, "w") as f:
    json.dump(payload, f)

print(f"wrote JSON: {dataset_path}   ({dataset_path.stat().st_size / 1024:.1f} KB)")


total cases: 80    per-case: N=512, features=5, targets=3
first case: aoa=5.95 deg,  Re=1242159.5
wrote JSON: /var/folders/dm/kyhk1vhx7jb_k1b2z7zdns6c0000gp/T/tmpc51d8g1z/synthetic_airfoil_slice.json   (5108.2 KB)


## 4. Log + promote to the Registry

The two-step pattern:

1. `run.log_artifact(...)` — creates a **project-level** artifact
   (versioned, but only visible inside this project).
2. `run.link_artifact(target_path="wandb-registry-{R}/{C}", aliases=[...])`
   — promotes that specific version into the Registry Collection.
   Org-visible, aliased, cross-project consumable. The Collection is
   created implicitly on first link; the Registry itself is created
   via `api.create_registry(...)` inside a `try/except` since
   `experiment_tracking.ipynb` may have created it already.

`log_artifact` is asynchronous — we call `.wait()` on the returned
reference before linking so the alias resolves on the backend.


In [4]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="dataset-registration",
    tags=["notebook", "registry", "dataset"],
    name="dataset-register-and-promote",
) as run:
    
    artifact = wandb.Artifact(
        name=DATASET_ARTIFACT_NAME,
        type="dataset",
        description="Synthetic NACA-4-digit airfoil cases (workshop dataset).",
        metadata={"n_cases": N_CASES, "n_points": N_POINTS, "seed": SEED},
    )
    artifact.add_file(str(dataset_path))
    logged = run.log_artifact(artifact)
    logged.wait()                                 # ensure upload commits



    api = wandb.Api()
    try:
        registry = api.create_registry(
            name=REGISTRY_NAME,
            visibility="organization",
        )
        print(f"created registry: {registry.name}")
    except Exception as exc:
        print(f"registry create skipped (may already exist): {exc}")

    target_path = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}"
    run.link_artifact(
        logged,
        target_path=target_path,
        aliases=["production", "v1"],
    )
    
    print(f"Dataset: {DATASET_ARTIFACT_NAME}")
    print(f"linked -> {target_path}   aliases: production, v1")
    print(f"run url: {run.url}")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jbutt (wandb_emea) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Artifact "synthetic_airfoil_slice" already exists with the same content. No new version will be created.


registry create skipped (may already exist): Registry 'airfoil_surrogates' already exists in organization 'wandb_emea', please use a different name.
Dataset: synthetic_airfoil_slice
linked -> wandb-registry-airfoil_surrogates/datasets   aliases: production, v1
run url: https://wandb.ai/wandb_emea/jb_physics_demo/runs/jn989136


## 5. Verify — read the dataset back from the Registry

Sanity-check: a fresh downstream run consumes the promoted entry.
This is exactly what `experiment_tracking.ipynb` and `sweeps.ipynb`
now do at their setup step.


In [5]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="registry-verify",
    tags=["notebook", "registry", "verify"],
    name="dataset-verify",
) as run:
    art = run.use_artifact(
        f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
    )
    download_dir = Path(art.download())
    json_file = next(download_dir.glob("*.json"))
    with open(json_file) as f:
        loaded = json.load(f)

    print(f"pulled {json_file.name}   "
          f"({json_file.stat().st_size / 1024:.1f} KB, "
          f"{len(loaded['cases'])} cases)")
    print(f"meta: {loaded['meta']}")


wandb:   1 of 1 files downloaded.  


pulled synthetic_airfoil_slice.json   (5108.2 KB, 80 cases)
meta: {'n_cases': 80, 'n_points': 512, 'seed': 42, 'generator': 'synthetic NACA-4-digit airfoils'}


## 6. Wrap up — what to look at in the W&B UI

**Runs** (two)
- `dataset-register-and-promote` (job_type = `dataset-registration`)
- `dataset-verify` (job_type = `registry-verify`)

**Registry**
- `wandb-registry-airfoil_surrogates/datasets` now contains one
  version of `synthetic_airfoil_slice`, aliased `production` + `v1`.

**Lineage graph**
- `synthetic_airfoil_slice:v0` produced by the register-run and
  consumed by the verify-run.

**Downstream** — `experiment_tracking.ipynb` and `sweeps.ipynb` both
resolve the dataset at
`wandb-registry-airfoil_surrogates/datasets:production` at their
setup step. Run those notebooks next.
